# DocMind · 02 — Fine-tune the answer model (Qwen2.5-7B-Instruct, QLoRA)

Fine-tunes **Qwen2.5-7B-Instruct** into DocMind's answer model and exports it as a **GGUF** file that
Ollama serves locally. The model is trained on DocMind's *exact* prompt format — the same system prompt,
`[CHUNK n | source: FILE | page P]` passages and `Sources:` section that `generation.py` sends — so what it
learns transfers directly.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. About 2 hours: ~10 min setup, ~90 min training,
~20 min GGUF export (a T4 trains a 7B model at only ~340 tokens/s, which is what `SCALE` trades against). Checkpoints are written to Google Drive, so a disconnected session can resume.

**Output (in Google Drive → `MyDrive/docmind/`):**
- `docmind-qwen2.5-7b.Q4_K_M.gguf` (~4.7 GB) — download it into the repo's `models/` folder, then
  `ollama create docmind-qwen -f training/Modelfile`
- `lora_adapter/` — the LoRA weights alone (~160 MB), a lightweight backup

### What the training data teaches
| Behaviour | Source | Why |
|---|---|---|
| Copy facts verbatim from the right passage | SQuAD 2.0 answerable, answering paragraph shuffled among same-article paragraphs | numbers, names and dates must come out exactly as written |
| Combine two passages | HotpotQA *distractor*: 2 supporting paragraphs plus 2–3 distractors | multi-hop questions |
| Refuse when the passages are on-topic but don't answer | SQuAD 2.0 unanswerable; HotpotQA with its supporting paragraphs removed | the hardest refusals — vocabulary overlaps, the answer is absent |
| Fluent answers, extraction and summaries | Dolly-15k `closed_qa`, `information_extraction`, `summarization` (with their passages) | natural answers and "what is this about?" questions |

Refusals are ~19% of the mix: enough to teach the behaviour without making the model refuse answerable
questions (the baseline's main failure was false refusals). **None of DocMind's gold-set documents are used** —
they stay test-only, so the local gold-set score measures generalisation, not memorisation.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import json, os, random, re, collections

SEED            = 3407
BASE_MODEL      = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # 4-bit copy of Qwen/Qwen2.5-7B-Instruct (Apache-2.0)
MAX_SEQ_LENGTH  = 2048
# Training mix (examples).  Scale all of them with SCALE to trade time for data.
# A T4 trains this 7B model at ~340 tokens/s, so SCALE=1.0 (~5,950 examples, mean ~1,250 tokens)
# takes ~6 h.  SCALE=0.33 (~1,960 examples, shorter contexts) takes ~1.5 h and is enough to learn
# the format, verbatim copying and refusal behaviour.
SCALE = 0.33
MIX = {
    "squad_answerable":   1500,
    "squad_unanswerable":  600,
    "hotpot_answerable":  1500,
    "hotpot_missing":      350,
    "dolly_grounded":     1300,
    "dolly_summary":       500,
    "dolly_mismatch":      200,
}
MIX = {k: int(v * SCALE) for k, v in MIX.items()}
DRIVE_DIR       = "/content/drive/MyDrive/docmind"
RUN_QUICK_EVAL  = True     # ~15 min: base vs fine-tuned on held-out SQuAD/HotpotQA validation questions

random.seed(SEED)
print(MIX, "total:", sum(MIX.values()))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

## 1 · DocMind's prompt format

Copied from `generation.py` (`SYSTEM_PROMPT` and `_build_user_prompt`). **Keep these in sync** — if the app's
prompt changes, re-run this notebook.

In [ ]:
SYSTEM_PROMPT = """\
You are a document lookup assistant. Your ONLY job is to answer questions \
using the text passages provided below — nothing else.

Rules you must follow without exception:
- Base your answer EXCLUSIVELY on the passages marked [CHUNK n | source: FILE].
- After your answer, list every source file you drew on under the heading \
"Sources:" — one file name per line, no duplicates.
- If the provided passages do not contain enough information to answer the \
question, respond with exactly: "I don't have enough information in the \
provided documents to answer this question." Do NOT speculate or use outside \
knowledge.
- Do not make up file names, page numbers, or details that are not in the \
passages.
- Keep your answer concise and factual.\
"""

REFUSAL = "I don't have enough information in the provided documents to answer this question."

def build_user_prompt(chunks, question):
    """chunks: list of (source_file, page, text) — same layout as generation._build_user_prompt."""
    blocks = []
    for i, (source, page, text) in enumerate(chunks, start=1):
        page_tag = f" | page {page}" if page else ""
        blocks.append(f"[CHUNK {i} | source: {source}{page_tag}]\n{text.strip()}")
    context = "\n\n---\n\n".join(blocks)
    return (f"PASSAGES:\n\n{context}\n\n---\n\nQUESTION: {question}\n\n"
            "Answer using only the passages above. List sources at the end.")

def answer_with_sources(answer, sources):
    return answer.strip() + "\n\nSources:\n" + "\n".join(dict.fromkeys(sources))

def as_file(title):
    return re.sub(r"[^\w\- ]+", "", title).strip().replace(" ", "_")[:60] + ".pdf"

def example(chunks, question, target):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(chunks, question)},
        {"role": "assistant", "content": target},
    ]}

def as_sentence(span):
    span = span.strip()
    return span if span.endswith((".", "!", "?")) else span + "."

## 2 · Build the training mix

In [ ]:
from datasets import load_dataset

squad  = load_dataset("rajpurkar/squad_v2")
hotpot = load_dataset("hotpotqa/hotpot_qa", "distractor")
dolly  = load_dataset("databricks/databricks-dolly-15k")["train"]

def squad_examples(split, n_answerable, n_unanswerable):
    by_title = collections.defaultdict(list)
    for ex in split:
        by_title[ex["title"]].append(ex)
    rows = [ex for exs in by_title.values() for ex in exs]
    random.shuffle(rows)
    out, need = [], {"a": n_answerable, "u": n_unanswerable}
    for ex in rows:
        kind = "a" if ex["answers"]["text"] else "u"
        if need[kind] == 0:
            continue
        contexts = list(dict.fromkeys(e["context"] for e in by_title[ex["title"]]))
        others = [c for c in contexts if c != ex["context"]]
        passages = [ex["context"]] + random.sample(others, min(len(others), random.randint(2, 4)))
        random.shuffle(passages)
        source = as_file(ex["title"])
        pages = random.sample(range(1, 40), len(passages))
        chunks = [(source, p, c) for p, c in zip(pages, passages)]
        target = answer_with_sources(as_sentence(ex["answers"]["text"][0]), [source]) if kind == "a" else REFUSAL
        out.append(example(chunks, ex["question"], target) | {"kind": f"squad_{'answerable' if kind == 'a' else 'unanswerable'}"})
        need[kind] -= 1
        if not any(need.values()):
            break
    return out

def hotpot_examples(split, n_answerable, n_missing):
    rows = list(range(len(split)))
    random.shuffle(rows)
    out, made_a, made_m = [], 0, 0
    for idx in rows:
        ex = split[idx]
        titles, sentences = ex["context"]["title"], ex["context"]["sentences"]
        gold = set(ex["supporting_facts"]["title"])
        paragraphs = [(as_file(t), random.randint(1, 30), " ".join(s).strip()) for t, s in zip(titles, sentences)]
        supporting = [p for p, t in zip(paragraphs, titles) if t in gold]
        distractors = [p for p, t in zip(paragraphs, titles) if t not in gold]
        # 2-3 distractors (not all 8) keeps prompts short and under MAX_SEQ_LENGTH, so the answer at the
        # end of the example isn't truncated away (Unsloth then drops the whole example).
        if made_a < n_answerable:
            chunks = supporting + random.sample(distractors, min(len(distractors), random.randint(2, 3)))
            random.shuffle(chunks)
            gold_files = [as_file(t) for t in titles if t in gold]
            target = answer_with_sources(as_sentence(ex["answer"]), gold_files)
            out.append(example(chunks, ex["question"], target) | {"kind": "hotpot_answerable"})
            made_a += 1
        elif made_m < n_missing:
            kept = random.sample(distractors, min(len(distractors), 4))
            out.append(example(kept, ex["question"], REFUSAL) | {"kind": "hotpot_missing"})
            made_m += 1
        else:
            break
    return out

def split_passage(text, max_words=180):
    # Long Dolly contexts become several chunks, like a DocMind document would.
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            chunks.append(" ".join(current)); current = []
        current.append(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

def dolly_examples(n_grounded, n_summary, n_mismatch):
    with_context = [r for r in dolly if r["context"].strip() and r["response"].strip()]
    grounded = [r for r in with_context if r["category"] in ("closed_qa", "information_extraction")]
    summaries = [r for r in with_context if r["category"] == "summarization"]
    random.shuffle(grounded); random.shuffle(summaries)
    out = []
    for i, r in enumerate(grounded[:n_grounded]):
        source = f"document_{i:04d}.pdf"
        own = [(source, p + 1, c) for p, c in enumerate(split_passage(r["context"]))]
        distractors = [(f"reference_{j:04d}.pdf", 1, d["context"]) for j, d in
                       enumerate(random.sample(with_context, 1))]
        chunks = own + distractors
        random.shuffle(chunks)
        out.append(example(chunks, r["instruction"], answer_with_sources(r["response"], [source])) | {"kind": "dolly_grounded"})
    for i, r in enumerate(summaries[:n_summary]):
        source = f"report_{i:04d}.pdf"
        chunks = [(source, p + 1, c) for p, c in enumerate(split_passage(r["context"]))]
        out.append(example(chunks, r["instruction"], answer_with_sources(r["response"], [source])) | {"kind": "dolly_summary"})
    unused = grounded[n_grounded:]
    for i, r in enumerate(unused[:n_mismatch]):
        # The question's own passage is withheld: only unrelated passages are shown.
        chunks = [(f"reference_{j:04d}.pdf", 1, d["context"]) for j, d in
                  enumerate(random.sample([x for x in with_context if x is not r], 3))]
        out.append(example(chunks, r["instruction"], REFUSAL) | {"kind": "dolly_mismatch"})
    return out

train_rows = (
    squad_examples(squad["train"], MIX["squad_answerable"], MIX["squad_unanswerable"])
    + hotpot_examples(hotpot["train"], MIX["hotpot_answerable"], MIX["hotpot_missing"])
    + dolly_examples(MIX["dolly_grounded"], MIX["dolly_summary"], MIX["dolly_mismatch"])
)
random.shuffle(train_rows)
kinds = collections.Counter(r["kind"] for r in train_rows)
refusals = sum(v for k, v in kinds.items() if k in ("squad_unanswerable", "hotpot_missing", "dolly_mismatch"))
print(kinds, f"\nrefusal share: {refusals / len(train_rows):.1%}")
print("\n--- sample ---\n", train_rows[0]["messages"][1]["content"][:1200], "\n>>>", train_rows[0]["messages"][2]["content"])

## 3 · Load Qwen2.5-7B-Instruct in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from datasets import Dataset

def render(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows]).map(render, remove_columns=["messages"])
lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"][:500]]
print(f"examples: {len(train_ds):,}  token length (sample of 500): mean {sum(lengths) / len(lengths):.0f}, "
      f"max {max(lengths)}, over {MAX_SEQ_LENGTH}: {sum(l > MAX_SEQ_LENGTH for l in lengths)}")

## 4 · Quick held-out check (before training)

Scores the **untuned** model on held-out SQuAD 2.0 / HotpotQA *validation* questions in DocMind's format, so
the same check after training shows what fine-tuning changed. (The real score is DocMind's own gold set, run
locally after export.)

In [ ]:
import torch

def normalize(s):
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(re.sub(r"[^a-z0-9. ]", " ", s).split())

def held_out(n_squad_a=60, n_squad_u=30, n_hotpot=60):
    random.seed(SEED + 1)
    rows = (squad_examples(squad["validation"], n_squad_a, n_squad_u)
            + hotpot_examples(hotpot["validation"], n_hotpot, 0))
    random.seed(SEED)
    return rows

def generate(messages, max_new_tokens=160):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def fits(r, margin=200):
    prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
    return len(tokenizer(prompt).input_ids) <= MAX_SEQ_LENGTH - margin

def quick_eval(rows, label):
    FastLanguageModel.for_inference(model)
    stats = collections.Counter()
    for r in [r for r in rows if fits(r)]:   # Unsloth refuses inputs longer than MAX_SEQ_LENGTH
        target = r["messages"][2]["content"]
        pred = generate(r["messages"])
        refused = "enough information" in pred.lower()
        if target == REFUSAL:
            stats["refusal_total"] += 1
            stats["refusal_ok"] += refused
        else:
            gold = normalize(target.split("\n\nSources:")[0]).rstrip(".")
            stats["answer_total"] += 1
            stats["answer_ok"] += (not refused) and gold in normalize(pred)
            stats["false_refusal"] += refused
    FastLanguageModel.for_training(model)
    print(f"[{label}] answer contains gold: {stats['answer_ok']}/{stats['answer_total']}  "
          f"false refusals: {stats['false_refusal']}  correct refusals: {stats['refusal_ok']}/{stats['refusal_total']}")
    return dict(stats)

eval_rows = held_out()
base_stats = quick_eval(eval_rows, "base") if RUN_QUICK_EVAL else None

## 5 · Train (loss on the assistant's answer only)

Checkpoints go to Drive every 100 steps. If the session drops, re-run the cells above (skip the quick eval)
and then this cell — it resumes from the last checkpoint.

In [ ]:
import glob
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

checkpoint_dir = f"{DRIVE_DIR}/checkpoints"
config_kwargs = dict(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    output_dir=checkpoint_dir,
    seed=SEED,
    report_to="none",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)
try:   # TRL renamed max_seq_length -> max_length; accept either version
    args = SFTConfig(max_seq_length=MAX_SEQ_LENGTH, **config_kwargs)
except TypeError:
    args = SFTConfig(max_length=MAX_SEQ_LENGTH, **config_kwargs)

try:   # TRL 1.x calls the tokenizer argument processing_class
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=args)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
resume = bool(glob.glob(f"{checkpoint_dir}/checkpoint-*"))
print("resuming from checkpoint" if resume else "starting fresh")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

In [ ]:
tuned_stats = quick_eval(eval_rows, "fine-tuned") if RUN_QUICK_EVAL else None
model.save_pretrained(f"{DRIVE_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{DRIVE_DIR}/lora_adapter")
with open(f"{DRIVE_DIR}/training_summary.json", "w") as f:
    json.dump({"base_model": BASE_MODEL, "mix": MIX, "examples": len(train_ds),
               "quick_eval": {"base": base_stats, "finetuned": tuned_stats}}, f, indent=2)

## 6 · Export to GGUF (Q4_K_M) for Ollama

Unsloth merges the LoRA weights into the base model and quantises with llama.cpp. Q4_K_M (~4.7 GB) fits an
8 GB GPU with room for an 8K context.

In [ ]:
import shutil

export_dir = "/content/gguf"
model.save_pretrained_gguf(export_dir, tokenizer, quantization_method="q4_k_m")
# Unsloth may write to a sibling folder (e.g. /content/gguf_gguf/), so search all of /content.
produced = sorted(glob.glob("/content/**/*.gguf", recursive=True), key=os.path.getsize)
print(produced)
gguf = [p for p in produced if "q4_k_m" in p.lower()][-1]
target = f"{DRIVE_DIR}/docmind-qwen2.5-7b.Q4_K_M.gguf"
shutil.copy(gguf, target)
print(f"saved {target} ({os.path.getsize(target) / 1e9:.2f} GB) — download it into the repo's models/ folder")